# TOPIK Master RAG: Concepts & Architecture

## Problem Statement
- Standard LLMs provide Korean explanations too complex for learners
- Incorrect honorifics for learner's specific level (1-6)
- Need: Level-aware retrieval of authentic TOPIK exam patterns

## Solution: Level-Aware RAG Pipeline
1. **Level-Specific Metadata**: Each document tagged with `topik_level` (1-6)
2. **Hybrid Retrieval**: BM25 (keyword) + Vector Similarity (semantic)
3. **Reranking**: BGE-Reranker-v2 for multilingual Korean content

## Key Formulas

### BM25 Score
```
score(D, Q) = Σ IDF(qi) * (f(qi, D) * (k1 + 1)) / (f(qi, D) + k1 * (1 - b + b * |D| / avgdl))
```
Where:
- `f(qi, D)`: term frequency of qi in document D
- `|D|`: document length
- `avgdl`: average document length in corpus
- `k1`, `b`: tuning parameters (typically 1.5, 0.75)

### Cosine Similarity (Vector Search)
```
similarity = (A · B) / (||A|| * ||B||)
```
Where A, B are embedding vectors

### Hybrid Score Combination
```
final_score = α * normalize(bm25_score) + (1 - α) * normalize(cosine_score)
```
Where α = 0.3 (configurable weighting)

## Sources & References
- [LangChain RAG Docs](https://python.langchain.com/docs/tutorials/rag/)
- [ChromaDB Documentation](https://docs.trychroma.com/)
- [BM25 Algorithm](https://en.wikipedia.org/wiki/Okapi_BM25)
- [BGE-Reranker Paper](https://arxiv.org/abs/2309.15088)
- [Ragas Evaluation Framework](https://docs.ragas.io/)
- [TOPIK Guide](https://www.topikguide.com/)

## Module Structure
```
src/
├── data/          # Ingestion (scraper, pdf_parser, level_tagger, ingest)
├── retrieval/     # Search (vector_store, bm25, reranker, embeddings, query)
├── evaluation/    # Metrics (ragas-based evaluation)
└── utils/         # Config, Logger
```

## How Each Block Works

### 1. Data Ingestion (`src/data/`)
- `scraper.py`: Web scrapes TOPIK vocabulary lists with level tags
- `pdf_parser.py`: Extracts text from past exam PDFs using PyMuPDF
- `level_tagger.py`: Auto-assigns TOPIK levels based on vocabulary complexity
- `ingest.py`: Orchestrates ingestion from multiple sources

### 2. Hybrid Retrieval (`src/retrieval/`)
- `vector_store.py`: ChromaDB persistent storage with level filtering
- `bm25.py`: Keyword-based retrieval for Korean particles (은/는, 이/가)
- `embeddings.py`: Korean embedding models (polyglot-ko-1.3b)
- `reranker.py`: BGE-Reranker-v2 for reranking retrieved documents
- `query.py`: Combines all retrievers into pipeline

### 3. Evaluation (`src/evaluation/`)
- `metrics.py`: Ragas metrics (Faithfulness, Context Precision)

### 4. Entry Points (`main/`)
- `ingest.py`: CLI for data ingestion
- `query.py`: CLI for level-filtered queries
- `eval.py`: CLI for Ragas evaluation
